In [1]:
## Deriving weights for clustering temperature zones based on predictor -> predictand relationship

# V1 - 2026-10-26 - for final run - JPK

# Workflow

In [2]:
# libraries
import os
import numpy as np
import pandas as pd
import rasterio
import rioxarray as rxr
import xarray as xr
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import gc
from pathlib import Path

# paths
sample_raster = rxr.open_rasterio(r"E:\ClimateData_Precipitation\Data\Datasets\Sample_raster\DEM_1km_elevation_max.tif")
clim_file_path = r"E:\ClimateData_Precipitation\Data\Processed\Spatial_clustering\Climatology_files"
clim_file_list = [f for f in os.listdir(clim_file_path) if f.endswith(".tif")]

# configurations
random_state_num = 64
n_cores = 6
target_var = 'prec_climatology'

In [3]:
# Stack the rasters 
data_arrays = {}
for file in clim_file_list:
    var_name = file.split(".")[0]
    da = rxr.open_rasterio(os.path.join(clim_file_path, file), masked=True)
    da = da.rio.reproject_match(sample_raster)
    
    if da.shape[0] == 1:
        da = da.squeeze('band', drop=True)
    
    data_arrays[var_name] = da

stacked_ds = xr.Dataset(data_arrays)
del data_arrays, sample_raster
gc.collect()

y_dim = stacked_ds.rio.y_dim  
x_dim = stacked_ds.rio.x_dim

valid_mask = None
for var in stacked_ds.data_vars:
    var_mask = ~np.isnan(stacked_ds[var].values)
    # add a nan filter for values = -9999
    valid_mask = var_mask if valid_mask is None else valid_mask & var_mask

stacked_ds_dict = {}
for var in stacked_ds.data_vars:
    stacked_ds_dict[var] = stacked_ds[var].values[valid_mask].flatten().astype('float32')

stacked_df = pd.DataFrame(stacked_ds_dict, dtype='float32')
del stacked_ds_dict, var_mask
gc.collect()

# Store spatial info
spatial_coords = {
    'x': stacked_ds[x_dim].values,
    'y': stacked_ds[y_dim].values,
    'crs': stacked_ds.rio.crs,
    'transform': stacked_ds.rio.transform(),
    'shape': (len(stacked_ds[y_dim]), len(stacked_ds[x_dim]))
}

del stacked_ds, valid_mask, y_dim, x_dim
gc.collect()

431

In [4]:
#3. Perform a simple random split

train_df, test_df = train_test_split(stacked_df, test_size=0.3, random_state=random_state_num)

X_train = train_df.drop(columns=[target_var])
y_train = train_df[target_var]

X_test = test_df.drop(columns=[target_var])
y_test = test_df[target_var]

feature_names = X_train.columns.tolist()

del stacked_df, train_df, test_df
gc.collect()

0

In [5]:
# 4. Fit random forest model

param_dist_rf = {
    'n_estimators': [100, 200, 300, 500], 
    'max_features': ['sqrt', 'log2'],
    'max_depth': [None, 10, 20, 30], 
    'min_samples_split': [2, 5, 10], 
    'min_samples_leaf': [1, 2, 4] 
}

rf = RandomForestRegressor(random_state=64)
rf_search = RandomizedSearchCV(
    estimator=rf, 
    param_distributions=param_dist_rf, 
    n_iter=50,  
    cv=5, 
    verbose=3, 
    random_state=random_state_num, 
    n_jobs= n_cores,
    scoring='neg_root_mean_squared_error'  
)

In [ ]:
# Fit model and calculate Permutation Feature Importance and convert to relative weights

rf_search.fit(X_train, y_train)
best_rf_model = rf_search.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits


NameError: name 't_var' is not defined

In [7]:
# 4. Final Validation Scores
print(f"\nBest RF Hyperparameters: {rf_search.best_params_}")
print(f"Best Mean CV Score (Train) : {rf_search.best_score_:.4f}")
print(f"Independent Test RMSE Score: {best_rf_model.score(X_test, y_test):.4f}")

pfi_test = permutation_importance(best_rf_model, X_test, y_test, scoring = 'neg_mean_absolute_error', random_state = random_state_num, n_repeats = 10, n_jobs = 1, max_samples = 0.25)

feature_importances_abs = dict(zip(feature_names, pfi_test.importances_mean))

#calculate relative importance 
feature_importances = {k: v / sum(feature_importances_abs.values()) for k, v in feature_importances_abs.items()}

#sort the dict by values in descending order
feature_importances = dict(sorted(feature_importances.items(), key=lambda item: item[1], reverse=True))
feature_importances_df = pd.DataFrame.from_dict(feature_importances, orient='index', columns=['importance'])
feature_importances_df.to_csv(fr'E:\ClimateData_Precipitation\Data\Processed\Spatial_clustering\climatology_weight_prec.csv')
print(feature_importances)



Best RF Hyperparameters: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}
Best Mean CV Score (Train) : -905.0163
Independent Test RMSE Score: 0.9935
{'GLDAS_specific_humidity_climatology': np.float64(0.153807512555509), 'GLDAS_wind_speed_climatology': np.float64(0.13002094542890713), 'GLDAS_lwdown_climatology': np.float64(0.11621120748592247), 'GLDAS_swdown_climatology': np.float64(0.09175292825429847), 'GLDAS_surface_pressure_climatology': np.float64(0.07839159202165939), 'GLDAS_air_temp_climatology': np.float64(0.06913937129950797), 'dem_smoothed': np.float64(0.0659022343309728), 'fetch_dist_500m_resampled': np.float64(0.05970583953175605), 'fetch_dist_1000m_resampled': np.float64(0.058425723637377046), 'MODIS_lst_n_climatology': np.float64(0.026860632599309386), 'DEM_1km_elevation_min': np.float64(0.021377694080429683), 'DEM_1km_elevation_max': np.float64(0.019742810070641992), 'DEM_1km_elevation_mean': np.float64(0.019